In [3]:
# Source - https://stackoverflow.com/a
# Posted by kcw78, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-25, License - CC BY-SA 4.0

import pandas as pd
import h5py


In [4]:
import numpy as np



file_in = r"C:\Users\Herman\Desktop\MFF\Diplomka\Data z PEMWE\091_I_I_Ir etched star, Pt 7 nm no C etched star\Data z PEMWE\I_Day1_procedure1_05_PEIS.mpt"
def get_header_lines(filepath):
    with open(filepath, "r", encoding="latin-1") as f:
        # Skip first line, read second
        f.readline()  
        line2 = f.readline().strip()
    # line2 looks like: "Nb header lines : 76"
    # split by ":" and take the right part
    try:
        n_header = int(line2.split(":")[1])
    except Exception:
        raise ValueError(f"Could not parse header line: {line2}")
    return n_header

def get_potential_control(file_path):
    with open(file_path, "r") as file:
        E = 'Ewe'
        for line_number, line in enumerate(file, start=1):
            if "Potential control" in line:
                # print(f"String found on line: {line_number}")
                # print(line.split(': '))
                if 'Ewe\n' in line.split(': ')[1]:
                    E = 'Ewe'
                elif 'Ewe-Ece\n' in line.split(': ')[1]:
                    E = 'Ecell'
    return E
print(get_potential_control(file_in))

# def get_regime(filepath):
#     with open(filepath, "r", encoding="latin-1") as f:
#         # Skip first line, read second
#         f.readline()  
#         line10 = f.readline().strip()
#     # line2 looks like: "Nb header lines : 76"
#     # split by ":" and take the right part
#     try:
#         n_header = int(line10.split(":")[9])
#     except Exception:
#         raise ValueError(f"Could not parse header line: {line10}")
#     return n_header



skip_rows = get_header_lines(file)
# print(skiprows)

with open(file, encoding='latin1') as f:
    df = pd.read_csv(f,skiprows = skip_rows-1,usecols=np.arange(0,30),delimiter = '\t')
print(df)

# print(data)

       freq/Hz  Re(Z)/Ohm  -Im(Z)/Ohm   |Z|/Ohm  Phase(Z)/deg        time/s  \
0     0.000000   0.000000    0.000000  0.000000      0.000000   4851.625287   
1     0.000000   0.000000    0.000000  0.000000      0.000000   4852.625288   
2     0.000000   0.000000    0.000000  0.000000      0.000000   4853.625288   
3     0.000000   0.000000    0.000000  0.000000      0.000000   4854.625288   
4     0.000000   0.000000    0.000000  0.000000      0.000000   4855.625288   
...        ...        ...         ...       ...           ...           ...   
3370  0.919118   0.033149    0.000460  0.033152     -0.795798  48245.442089   
3371  0.789940   0.033054   -0.000074  0.033054      0.128022  48250.798785   
3372  0.677639   0.033468    0.000098  0.033468     -0.167811  48257.015979   
3373  0.582023   0.033240    0.000194  0.033241     -0.334222  48264.226794   
3374  0.500352   0.033105    0.000231  0.033106     -0.399506  48272.587202   

       <Ewe>/V       <I>/mA         Cs/µF      Cp/µ

In [13]:
def save_df_to_h5_by_cycle(file_in, file_out, columns, cycle_col="cycle number", global_metadata=None):
    """
    df: pandas DataFrame with multiple spectra
    cycle_col: column that identifies each spectrum
    global_metadata: dict stored at file level (optional)
    """
    
    skip_rows = get_header_lines(file_in)
    with open(file_in, encoding='latin1') as f:
        df = pd.read_csv(f,skiprows = skip_rows-1,usecols=np.arange(0,30),delimiter = '\t')
    # print(df)
    with h5py.File(file_out, "w") as f:

        # Store global metadata
        if global_metadata is not None:
            for k, v in global_metadata.items():
                f.attrs[k] = v

        # Group by spectrum
        for cycle, group in df.groupby(cycle_col):
            grp = f.create_group(f"spectrum_{(int(cycle)):02d}")
            
            
            # Store each column as its own dataset
            for col in group.columns:
                # print(col)
                # if col == cycle_col:
                    # continue  # already encoded in group name
                if col in columns:
                    data = group[col].to_numpy()
                    col_split = col.split("/")
                    print(col_split)
                    grp.create_dataset(str(col_split[0]+' ('+col_split[1]+')'), data=data)

            # Optionally store per-spectrum metadata
            grp.attrs["cycle number"] = cycle

In [14]:
E = get_potential_control(file_in)
if E == 'Ewe':
    columns = ['freq/Hz', 'Re(Z)/Ohm','-Im(Z)/Ohm','<Ewe>/V','time/s','<I>/mA']
elif E == 'Ecell':
    columns = ['freq/Hz', 'Re(Zwe-ce)/Ohm','-Im(Zwe-ce)/Ohm','<Ewe>/V','time/s','<I>/mA']
file_in = r"C:\Users\Herman\Desktop\MFF\Diplomka\Data z PEMWE\091_I_I_Ir etched star, Pt 7 nm no C etched star\Data z PEMWE\I_Day1_procedure1_05_PEIS.mpt"

save_df_to_h5_by_cycle(file_in,'file_h5py2.h5', columns, cycle_col="cycle number", global_metadata=None)

       freq/Hz  Re(Z)/Ohm  -Im(Z)/Ohm   |Z|/Ohm  Phase(Z)/deg        time/s  \
0     0.000000   0.000000    0.000000  0.000000      0.000000   4851.625287   
1     0.000000   0.000000    0.000000  0.000000      0.000000   4852.625288   
2     0.000000   0.000000    0.000000  0.000000      0.000000   4853.625288   
3     0.000000   0.000000    0.000000  0.000000      0.000000   4854.625288   
4     0.000000   0.000000    0.000000  0.000000      0.000000   4855.625288   
...        ...        ...         ...       ...           ...           ...   
3370  0.919118   0.033149    0.000460  0.033152     -0.795798  48245.442089   
3371  0.789940   0.033054   -0.000074  0.033054      0.128022  48250.798785   
3372  0.677639   0.033468    0.000098  0.033468     -0.167811  48257.015979   
3373  0.582023   0.033240    0.000194  0.033241     -0.334222  48264.226794   
3374  0.500352   0.033105    0.000231  0.033106     -0.399506  48272.587202   

       <Ewe>/V       <I>/mA         Cs/µF      Cp/µ

In [20]:
# search_string = 
# file_path = file_in
def get_potential_control(file_path):
    with open(file_path, "r") as file:
        E = 'Ewe'
        for line_number, line in enumerate(file, start=1):
            if "Potential control" in line:
                # print(f"String found on line: {line_number}")
                # print(line.split(': '))
                if 'Ewe\n' in line.split(': ')[1]:
                    E = 'Ewe'
                elif 'Ewe-Ece\n' in line.split(': ')[1]:
                    E = 'Ecell'
    return E
print(get_potential_control(file_in))

Ewe
